# EDA - `player_appearance_behaviour_under_pressure.csv`

Lean EDA of the under-pressure event table - on-ball events captured while a
player is being pressed by an opponent. **Not used to derive any column in
the main panel**, so every feature engineered below is genuinely new and
directly addresses RQ4 (does pressure data add to the goal-scoring model?).

Two deliverables, mirroring the run-EDA structure:

1. **Data cleaning** - structural integrity, NULL semantics, domain bounds.
2. **Feature manifest** - which engineered candidates carry signal.

Section structure:

| ID | Section |
|----|---------|
| A | Data cleaning |
| B | Engineer candidate features at (player_appearance_id, checkpoint) |
| C | Bivariate signal vs `scored_after` (pooled + by position) |
| D | Redundancy + cluster-robust significance |
| E | Final manifest |


## Setup

In [1]:
"""Imports."""
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
"""Display options."""
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
pd.set_option("display.precision", 4)


In [3]:
"""Paths and load."""
DATA_DIR = PROJECT_ROOT / "data"
PRESS_PATH = DATA_DIR / "player_appearance_behaviour_under_pressure.csv"
MAIN_PATH = DATA_DIR / "players_quarters_final.csv"

# `pass_angle` arrives as quoted strings -> coerce. NULL stays NaN.
press = pd.read_csv(PRESS_PATH)
press["pass_angle"] = pd.to_numeric(press["pass_angle"], errors="coerce")

main = pd.read_csv(MAIN_PATH)

appearance_meta = (
    main[["player_appearance_id", "player_id", "fixture_id", "position",
          "is_home", "minute_in", "minute_out"]]
    .drop_duplicates(subset="player_appearance_id")
    .set_index("player_appearance_id")
)
press.shape, main.shape


((12185, 10), (3486, 33))

In [4]:
"""Helpers (Wilson CI + binned-rate table)."""
from scipy import stats as _stats


def wilson_ci(successes: int, trials: int, alpha: float = 0.05) -> tuple[float, float]:
    if trials == 0:
        return float("nan"), float("nan")
    p = successes / trials
    z = _stats.norm.ppf(1.0 - alpha / 2.0)
    denom = 1.0 + (z ** 2) / trials
    centre = (p + (z ** 2) / (2.0 * trials)) / denom
    half = z * np.sqrt(p * (1.0 - p) / trials + (z ** 2) / (4.0 * trials ** 2)) / denom
    return float(centre - half), float(centre + half)


def binned_rate(df: pd.DataFrame, feature: str, target: str = "scored_after",
                bins=None, by: str | None = None) -> pd.DataFrame:
    work = df[[feature, target] + ([by] if by else [])].copy()
    if bins is not None:
        work["__bin"] = pd.cut(work[feature], bins=bins, include_lowest=True)
    else:
        work["__bin"] = work[feature]
    grouping = ["__bin"] if by is None else [by, "__bin"]
    g = work.groupby(grouping, observed=False, dropna=False)[target].agg(["size", "sum"])
    g = g.rename(columns={"size": "n", "sum": "n_pos"}).reset_index()
    g["rate"] = g["n_pos"] / g["n"].where(g["n"] > 0)
    ci = g.apply(lambda r: wilson_ci(int(r["n_pos"]), int(r["n"])), axis=1, result_type="expand")
    g["ci_lo"], g["ci_hi"] = ci[0], ci[1]
    return g.rename(columns={"__bin": "bin"})


PERIOD_ORDER = {"half_1": 0, "half_2": 1, "extra_time_1": 2, "extra_time_2": 3}
CHECKPOINTS = [
    ("H1_15", "half_1", 15), ("H1_30", "half_1", 30), ("H1_45", "half_1", 45),
    ("H2_15", "half_2", 15), ("H2_30", "half_2", 30), ("H2_45", "half_2", 45),
    ("ET1_15", "extra_time_1", 15),
]
CP_ORDER = {cp: PERIOD_ORDER[per] for cp, per, _ in CHECKPOINTS}
CP_MINUTE = {cp: m for cp, _, m in CHECKPOINTS}
CP_PERIOD = {cp: per for cp, per, _ in CHECKPOINTS}
MATCH_MINUTE = {"H1_15":15, "H1_30":30, "H1_45":45,
                "H2_15":60, "H2_30":75, "H2_45":90, "ET1_15":105}


## Section A - Data cleaning

| ID | Check |
|----|-------|
| A1 | Shape, dtypes, head |
| A2 | PK uniqueness on `id` |
| A3 | Missingness pattern (`pass_angle` is NULL by design for turnover / ball_carry; `addressee_player_appearance_id` similarly) |
| A4 | Domain values for `period`, `press_induced_outcome`, `stage`, `accurate` |
| A5 | `pass_angle` bounds and NULL coherence with `press_induced_outcome` |
| A6 | Stoppage-time encoding |
| A7 | Orphan appearances (player_appearance_id absent from main) - both for the *pressed* player and for the *pressing* player |


### A1 - Shape & dtypes

In [24]:
print("shape:", press.shape)
print()
print(press.dtypes.to_string())
press.head(3)


shape: (12185, 10)

id                                  int64
period                             object
player_appearance_id                int64
addressee_player_appearance_id    float64
accurate                             bool
pressing_player_appearance_id       int64
press_induced_outcome              object
pass_angle                        float64
minute                              int64
stage                              object


,id,period,player_appearance_id,addressee_player_appearance_id,accurate,pressing_player_appearance_id,press_induced_outcome,pass_angle,minute,stage
0,244091,half_1,40764,NaN,False,40788,turnover,NaN,3,bottom
1,244092,half_1,40764,40761.0,True,40788,backward_pass,27.526,3,bottom
2,244093,half_1,40764,40781.0,False,40788,turnover,NaN,3,middle


### A2 - PK uniqueness on `id`

In [25]:
dupes = press["id"].duplicated().sum()
print(f"duplicate `id` rows: {dupes}")
print(f"unique ids: {press['id'].nunique()} / {len(press)}")


duplicate `id` rows: 0
unique ids: 12185 / 12185


### A3 - Missingness

In [26]:
na = press.isna().sum()
print("columns with NaN counts:")
print(na.to_string())
print()
print("expected NULL semantics:")
print(" - pass_angle: NULL whenever press_induced_outcome in {turnover, ball_carry}")
print(" - addressee_player_appearance_id: NULL on turnover (no target)")


columns with NaN counts:
id                                   0
period                               0
player_appearance_id                 0
addressee_player_appearance_id     649
accurate                             0
pressing_player_appearance_id        0
press_induced_outcome                0
pass_angle                        5793
minute                               0
stage                               14

expected NULL semantics:
 - pass_angle: NULL whenever press_induced_outcome in {turnover, ball_carry}
 - addressee_player_appearance_id: NULL on turnover (no target)


### A4 - Domain values

In [27]:
for c in ("period", "press_induced_outcome", "stage", "accurate"):
    vc = press[c].value_counts(dropna=False)
    print(f"{c}:")
    print(vc.to_string())
    print()


period:
period
half_1          7217
half_2          4724
extra_time_1     138
extra_time_2     106

press_induced_outcome:
press_induced_outcome
turnover         4763
backward_pass    3213
forward_pass     2274
ball_carry       1030
lateral_pass      905

stage:
stage
middle    5492
top       4076
bottom    2603
NaN         14

accurate:
accurate
True     6950
False    5235



### A5 - `pass_angle` bounds + NULL coherence

Two checks:

1. The angle is reported in degrees - what range does the data actually
   use? Common conventions are `(-180, 180]` (signed) or `[0, 360)`
   (unsigned).
2. By data-dictionary contract, `pass_angle` should be NULL exactly when
   `press_induced_outcome` is `turnover` or `ball_carry` (no pass made).
   Any deviation flags an upstream pipeline issue.


In [28]:
print(f"pass_angle range: [{press['pass_angle'].min():.2f}, {press['pass_angle'].max():.2f}]")
print(f"pass_angle NaN: {press['pass_angle'].isna().sum()}")
print()
print("NULL coherence with press_induced_outcome:")
ct = pd.crosstab(press["press_induced_outcome"], press["pass_angle"].isna(), margins=True)
ct.columns = ["pass_angle_present", "pass_angle_NULL", "total"]
print(ct.to_string())


pass_angle range: [-89.82, 89.99]
pass_angle NaN: 5793

NULL coherence with press_induced_outcome:
                       pass_angle_present  pass_angle_NULL  total
press_induced_outcome                                            
backward_pass                        3213                0   3213
ball_carry                              0             1030   1030
forward_pass                         2274                0   2274
lateral_pass                          905                0    905
turnover                                0             4763   4763
All                                  6392             5793  12185


### A6 - Stoppage-time encoding

In [29]:
cap = {"half_1": 45, "half_2": 45, "extra_time_1": 15, "extra_time_2": 15}
press["__cap"] = press["period"].map(cap)
stoppage = press[press["minute"] > press["__cap"]]
print(f"stoppage-time press events: {len(stoppage)} ({len(stoppage)/len(press):.2%})")
print()
print("by period:")
print(stoppage.groupby("period")["minute"].agg(["count", "min", "max"]).to_string())
press.drop(columns="__cap", inplace=True)


stoppage-time press events: 739 (6.06%)

by period:
              count  min  max
period                       
extra_time_1      5   16   16
extra_time_2     11   16   18
half_1          269   46   50
half_2          454   46   56


### A7 - Orphan appearances

Two FK columns reference player_appearance.id: the *pressed* player
(`player_appearance_id`) and the *pressing* player
(`pressing_player_appearance_id`). Either may reference a late substitute
who never crossed a checkpoint and is therefore absent from `main`.


In [30]:
main_apps = set(main["player_appearance_id"].unique())
press_apps = set(press["player_appearance_id"].unique())
pressing_apps = set(press["pressing_player_appearance_id"].dropna().astype(int).unique())

orphan_pressed = press_apps - main_apps
orphan_pressing = pressing_apps - main_apps

print(f"distinct *pressed* appearances in press: {len(press_apps)}")
print(f"distinct *pressing* appearances in press: {len(pressing_apps)}")
print(f"distinct appearances in main:           {len(main_apps)}")
print(f"orphan (pressed - main):  {len(orphan_pressed)}  ({press['player_appearance_id'].isin(orphan_pressed).sum()} rows)")
print(f"orphan (pressing - main): {len(orphan_pressing)} ({press['pressing_player_appearance_id'].isin(orphan_pressing).sum()} rows)")


distinct *pressed* appearances in press: 923
distinct *pressing* appearances in press: 895
distinct appearances in main:           869
orphan (pressed - main):  79  (199 rows)
orphan (pressing - main): 88 (254 rows)


### Section A - findings

Decisions logged for Section B:

* Filter press events to `player_appearance_id in main` (covers the pressed
  player; the pressing-player FK is used only as a count, so an orphan
  pressing FK is fine).
* `pass_angle` NULL only when `press_induced_outcome in {turnover, ball_carry}`
  (verify this holds; if not, treat the pattern as data-quality rather
  than feature-engineering).
* Stoppage-time rows carried over (apply same windowing rule as runs / shots).
* `pass_angle` will be feature-engineered as both a **signed** (vertical
  vs lateral direction) and an **absolute** (deviation from straight)
  feature.


## Section B - Engineered candidate features

Since the main panel carries **zero** under-pressure features, every
candidate below is new. Each is computed at the (player_appearance_id,
checkpoint) grain using the strict windowing rule from earlier EDAs.

| # | Feature | Formula | Rationale |
|---|---------|---------|-----------|
| 1 | `cumul_press_events`         | count | how often the player was pressed (volume) |
| 2 | `last15_press_events`        | count | recent pressing intensity |
| 3 | `cumul_press_turnovers`      | count | losses under pressure |
| 4 | `press_turnover_rate`        | turnovers / events | composure failure rate |
| 5 | `press_accuracy`             | accurate / events_with_addressee | composure success rate |
| 6 | `forward_pass_share`         | forward / (forward+backward+lateral) | progressive intent under pressure |
| 7 | `cumul_press_top_third`      | count of press events in attacking third | spatial threat under pressure |
| 8 | `top_third_press_share`      | share of press events in attacking third | spatial concentration |
| 9 | `cumul_pressing_others`      | count of times THIS player applied pressure | tactical role - high-press vs deep-block |
| 10 | `mean_abs_pass_angle`       | mean of `|pass_angle|` (passes only) | how laterally vs vertically the player plays under pressure |
| 11 | `last15_press_uplift`       | recent press rate vs cumul press rate | RQ6 anchor for this domain |
| 12 | `pressing_minus_pressed`    | `cumul_pressing_others - cumul_press_events` | net pressure direction (does the player apply more than they receive?) |


In [31]:
"""Filter to the main panel and tag period_order."""
press_clean = press[press["player_appearance_id"].isin(main_apps)].copy()
press_clean["period_order"] = press_clean["period"].map(PERIOD_ORDER)
print(f"press_clean shape: {press_clean.shape}")

# We also need rows where THIS player was the one pressing - that uses
# pressing_player_appearance_id as the join key. Build a mirror view.
press_as_presser = press[press["pressing_player_appearance_id"].isin(main_apps)].copy()
press_as_presser["period_order"] = press_as_presser["period"].map(PERIOD_ORDER)
print(f"press_as_presser shape: {press_as_presser.shape}")


press_clean shape: (11986, 11)
press_as_presser shape: (11931, 11)


In [32]:
"""Aggregate at checkpoint level - cumul + last15.

Two parallel pipelines:
 * `pressed`     - rows joined on player_appearance_id (the *pressed* player)
 * `pressing`    - rows joined on pressing_player_appearance_id (this player applied pressure)
"""
def _agg_pressed(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("player_appearance_id")
    out = pd.DataFrame({
        "events": g.size(),
        "turnovers": g.apply(lambda x: (x["press_induced_outcome"] == "turnover").sum()),
        "forward_passes": g.apply(lambda x: (x["press_induced_outcome"] == "forward_pass").sum()),
        "backward_passes": g.apply(lambda x: (x["press_induced_outcome"] == "backward_pass").sum()),
        "lateral_passes": g.apply(lambda x: (x["press_induced_outcome"] == "lateral_pass").sum()),
        "ball_carries": g.apply(lambda x: (x["press_induced_outcome"] == "ball_carry").sum()),
        "accurate_passes": g.apply(lambda x: (x["accurate"] == True).sum()),
        "events_with_addressee": g.apply(lambda x: x["addressee_player_appearance_id"].notna().sum()),
        "top_third": g.apply(lambda x: (x["stage"] == "top").sum()),
        "abs_angle_sum": g.apply(lambda x: x["pass_angle"].abs().sum()),
        "abs_angle_n":   g.apply(lambda x: x["pass_angle"].notna().sum()),
    })
    return out


def _agg_pressing(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("pressing_player_appearance_id")
    out = pd.DataFrame({"pressing_others": g.size()})
    out.index.name = "player_appearance_id"
    return out


def panel_for_cp(cp: str) -> pd.DataFrame:
    cp_per = CP_PERIOD[cp]
    cp_min = CP_MINUTE[cp]
    cp_ord = CP_ORDER[cp]

    cumul_mask = (
        (press_clean["period_order"] < cp_ord) |
        ((press_clean["period_order"] == cp_ord) & (press_clean["minute"] <= cp_min))
    )
    last15_mask = (
        (press_clean["period"] == cp_per) &
        (press_clean["minute"] > cp_min - 15) &
        (press_clean["minute"] <= cp_min)
    )

    cumul = _agg_pressed(press_clean[cumul_mask]).add_prefix("cumul_")
    last15 = _agg_pressed(press_clean[last15_mask]).add_prefix("last15_")

    cumul_p = (press_as_presser["period_order"] < cp_ord) | (
        (press_as_presser["period_order"] == cp_ord) & (press_as_presser["minute"] <= cp_min)
    )
    pressing_cumul = _agg_pressing(press_as_presser[cumul_p]).add_prefix("cumul_")

    out = cumul.join(last15, how="outer").join(pressing_cumul, how="outer").fillna(0)
    out["checkpoint"] = cp
    return out.reset_index()


panel = pd.concat([panel_for_cp(cp) for cp, _, _ in CHECKPOINTS], ignore_index=True)
panel.shape


C:\Users\tymot\AppData\Local\Temp\ipykernel_18928\3580686308.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "turnovers": g.apply(lambda x: (x["press_induced_outcome"] == "turnover").sum()),
C:\Users\tymot\AppData\Local\Temp\ipykernel_18928\3580686308.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "forward_passes": g.apply(lambda x: (x["press_induced_outcome"] == "forward_pass").sum()),
C:\Users\

(5172, 25)

In [33]:
"""Compose the 12 candidate features."""
EPS = 1e-6

# Time / exposure context, identical to the run / cross EDAs.
am = appearance_meta.reset_index()[["player_appearance_id", "minute_in", "minute_out"]]
panel = panel.merge(am, on="player_appearance_id", how="left")
panel["match_minute_at_cp"] = panel["checkpoint"].map(MATCH_MINUTE)
panel["mins_played_so_far"] = (
    panel[["match_minute_at_cp", "minute_out"]].min(axis=1)
    - panel["minute_in"].clip(lower=1) + 1
).clip(lower=1)

feat = pd.DataFrame({
    "player_appearance_id": panel["player_appearance_id"],
    "checkpoint": panel["checkpoint"],
})
feat["cumul_press_events"]    = panel["cumul_events"]
feat["last15_press_events"]   = panel["last15_events"]
feat["cumul_press_turnovers"] = panel["cumul_turnovers"]
feat["press_turnover_rate"]   = panel["cumul_turnovers"] / panel["cumul_events"].replace(0, np.nan)
feat["press_accuracy"]        = panel["cumul_accurate_passes"] / panel["cumul_events_with_addressee"].replace(0, np.nan)

passes_directional = (
    panel["cumul_forward_passes"]
    + panel["cumul_backward_passes"]
    + panel["cumul_lateral_passes"]
).replace(0, np.nan)
feat["forward_pass_share"]    = panel["cumul_forward_passes"] / passes_directional

feat["cumul_press_top_third"] = panel["cumul_top_third"]
feat["top_third_press_share"] = panel["cumul_top_third"] / panel["cumul_events"].replace(0, np.nan)
feat["cumul_pressing_others"] = panel["cumul_pressing_others"]
feat["mean_abs_pass_angle"]   = panel["cumul_abs_angle_sum"] / panel["cumul_abs_angle_n"].replace(0, np.nan)

last15_rate = panel["last15_events"] / 15.0
cumul_rate  = panel["cumul_events"] / panel["mins_played_so_far"]
feat["last15_press_uplift"]   = last15_rate / (cumul_rate + EPS)
feat["pressing_minus_pressed"] = panel["cumul_pressing_others"] - panel["cumul_events"]

feat = feat.replace([np.inf, -np.inf], np.nan)
feat["last15_press_uplift"] = feat["last15_press_uplift"].clip(upper=10)
feat.describe().T[["count", "mean", "std", "min", "50%", "max"]]


,count,mean,std,min,50%,max
player_appearance_id,5172.0,43354.0820,7345.5085,39422.0000,41256.0000,65789.0000
cumul_press_events,5172.0,10.8908,8.0893,0.0000,9.0000,55.0000
last15_press_events,5172.0,2.1723,2.4815,0.0000,1.0000,28.0000
cumul_press_turnovers,5172.0,4.2224,3.5812,0.0000,3.0000,24.0000
press_turnover_rate,5070.0,0.3955,0.2333,0.0000,0.3750,1.0000
press_accuracy,5053.0,0.5992,0.2392,0.0000,0.6190,1.0000
forward_pass_share,4723.0,0.3669,0.2900,0.0000,0.3333,1.0000
cumul_press_top_third,5172.0,3.4321,4.1892,0.0000,2.0000,30.0000
top_third_press_share,5070.0,0.2825,0.2649,0.0000,0.2462,1.0000
cumul_pressing_others,5172.0,10.8699,8.6118,0.0000,9.0000,66.0000


### B - findings

Sanity checks on the engineered features:

* `cumul_press_events` should grow monotonically through checkpoints; the
  median value at H2_30 is the natural reference for "typical pressure
  exposure across a full match".
* `press_turnover_rate` and `press_accuracy` should be roughly
  complementary; if both are high or both low for the same player, one of
  the denominators (events vs events_with_addressee) is subtly different.
* `forward_pass_share` ranges in [0, 1]; a player with 0 means they
  recycled or carried under every press; 1 means every pass under
  pressure went forward.
* `last15_press_uplift` is undefined when `cumul_press_events == 0` (zero
  prior pressure) - this hits early-checkpoint rows and is the expected
  NaN pattern.


## Section C - Bivariate signal vs `scored_after`

Per feature: pooled binned target rate with Wilson 95% CI. Stratification
by position is added on the strongest candidates only (saves notebook
space; if a feature is flat pooled, position rarely rescues it for this
domain).


In [34]:
"""Merge for analysis."""
m = main[["player_appearance_id", "checkpoint", "scored_after",
          "fixture_id", "position"]].merge(
    feat, on=["player_appearance_id", "checkpoint"], how="left"
)
BASELINE = m["scored_after"].mean()
print(f"baseline scored_after rate: {BASELINE:.4f}")


baseline scored_after rate: 0.0582


In [35]:
"""Pearson r vs target."""
NUMERIC = [c for c in feat.columns if c not in ("player_appearance_id", "checkpoint")]
rows = []
for c in NUMERIC:
    s = m[c].dropna()
    if s.std() == 0:
        rows.append({"feature": c, "n": len(s), "pearson_r": float("nan")})
        continue
    t = m.loc[s.index, "scored_after"]
    rows.append({"feature": c, "n": len(s), "pearson_r": float(np.corrcoef(s, t)[0, 1])})
pearson_df = pd.DataFrame(rows).sort_values(
    "pearson_r", key=lambda x: x.abs(), ascending=False
).reset_index(drop=True)
pearson_df


,feature,n,pearson_r
0,top_third_press_share,3238,0.1075
1,press_accuracy,3225,-0.0802
2,press_turnover_rate,3238,0.0694
3,last15_press_events,3324,0.0518
4,cumul_press_top_third,3324,0.0499
5,mean_abs_pass_angle,2992,-0.0421
6,forward_pass_share,2992,-0.0376
7,cumul_press_turnovers,3324,0.0352
8,pressing_minus_pressed,3324,0.0287
9,last15_press_uplift,3324,0.0268


In [36]:
"""Binned target rates."""
BINS = {
    "cumul_press_events":     [-1, 0, 5, 15, 200],
    "last15_press_events":    [-1, 0, 2, 5, 50],
    "cumul_press_turnovers":  [-1, 0, 1, 3, 50],
    "press_turnover_rate":    [-0.01, 0.0, 0.2, 0.5, 1.01],
    "press_accuracy":         [-0.01, 0.0, 0.5, 0.8, 1.01],
    "forward_pass_share":     [-0.01, 0.0, 0.25, 0.5, 1.01],
    "cumul_press_top_third":  [-1, 0, 2, 5, 100],
    "top_third_press_share":  [-0.01, 0.0, 0.2, 0.5, 1.01],
    "cumul_pressing_others":  [-1, 0, 3, 8, 100],
    "mean_abs_pass_angle":    [-0.01, 0.0, 30, 60, 181],
    "last15_press_uplift":    [-1, 0, 0.5, 1.5, 11],
    "pressing_minus_pressed": [-50, -3, 0, 3, 50],
}

frames = []
for f, edges in BINS.items():
    t = binned_rate(m, f, "scored_after", bins=edges)
    t["feature"] = f
    frames.append(t)
binned_df = pd.concat(frames, ignore_index=True)[
    ["feature", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]
]
binned_df


,feature,bin,n,n_pos,rate,ci_lo,ci_hi
0,cumul_press_events,"(-1.001, 0.0]",86,8,0.0930,0.0479,0.1730
1,cumul_press_events,"(0.0, 5.0]",1133,77,0.0680,0.0547,0.0841
2,cumul_press_events,"(5.0, 15.0]",1499,82,0.0547,0.0443,0.0674
3,cumul_press_events,"(15.0, 200.0]",606,32,0.0528,0.0377,0.0736
4,cumul_press_events,NaN,162,4,0.0247,0.0096,0.0618
5,last15_press_events,"(-1.001, 0.0]",494,22,0.0445,0.0296,0.0665
6,last15_press_events,"(0.0, 2.0]",1186,60,0.0506,0.0395,0.0646
7,last15_press_events,"(2.0, 5.0]",1135,78,0.0687,0.0554,0.0849
8,last15_press_events,"(5.0, 50.0]",509,39,0.0766,0.0566,0.1030
9,last15_press_events,NaN,162,4,0.0247,0.0096,0.0618


In [37]:
"""Stratified by position - top 3 features by |r| only."""
top3 = pearson_df.head(3)["feature"].tolist()
print(f"stratifying: {top3}")
sub = m[m["position"].isin(["A", "M", "D"])].copy()
strat_frames = []
for f in top3:
    t = binned_rate(sub, f, "scored_after", bins=BINS[f], by="position")
    t["feature"] = f
    strat_frames.append(t)
strat_df = pd.concat(strat_frames, ignore_index=True)[
    ["feature", "position", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]
]
strat_df


stratifying: ['top_third_press_share', 'press_accuracy', 'press_turnover_rate']


,feature,position,bin,n,n_pos,rate,ci_lo,ci_hi
0,top_third_press_share,A,"(-0.011, 0.0]",70,8,0.1143,0.0591,0.2096
1,top_third_press_share,A,"(0.0, 0.2]",50,4,0.0800,0.0315,0.1884
2,top_third_press_share,A,"(0.2, 0.5]",268,30,0.1119,0.0795,0.1553
3,top_third_press_share,A,"(0.5, 1.01]",204,29,0.1422,0.1008,0.1967
4,top_third_press_share,A,NaN,23,6,0.2609,0.1255,0.4647
5,top_third_press_share,D,"(-0.011, 0.0]",649,28,0.0431,0.0300,0.0616
6,top_third_press_share,D,"(0.0, 0.2]",210,2,0.0095,0.0026,0.0341
7,top_third_press_share,D,"(0.2, 0.5]",257,2,0.0078,0.0021,0.0279
8,top_third_press_share,D,"(0.5, 1.01]",84,3,0.0357,0.0122,0.0998
9,top_third_press_share,D,NaN,46,3,0.0652,0.0224,0.1750


### C - findings

Read off the four tables above. Patterns to watch for:

* **`forward_pass_share`** - a player consistently playing forward under
  pressure is showing attacking intent. Expect a positive monotone
  relationship.
* **`top_third_press_share`** - mirrors the strongest signal from runs
  EDA (top-third concentration). Expect this to be the single strongest
  feature in this table.
* **`press_turnover_rate`** - high turnover rate = sloppy under pressure;
  expect a *negative* relationship with `scored_after`.
* **`cumul_pressing_others`** - applying pressure is a deep-block /
  defensive-midfielder signal; expect this to be flat or slightly
  negative.
* **`pressing_minus_pressed`** - net direction. Negative = pressed more
  than pressing (attacking team's playmaker?), positive = pressing more
  (defensive role).


## Section D - Redundancy + cluster-robust significance

In [38]:
"""Pearson correlation matrix."""
corr = m[NUMERIC].corr(method="pearson")
hi = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .rename("pearson")
        .reset_index()
        .rename(columns={"level_0": "a", "level_1": "b"})
)
hi["abs"] = hi["pearson"].abs()
hi.sort_values("abs", ascending=False).head(15).reset_index(drop=True)


,a,b,pearson,abs
0,press_turnover_rate,press_accuracy,-0.9310,0.9310
1,cumul_press_events,cumul_press_turnovers,0.8171,0.8171
2,cumul_press_events,cumul_press_top_third,0.6951,0.6951
3,cumul_press_top_third,top_third_press_share,0.6432,0.6432
4,cumul_press_turnovers,cumul_press_top_third,0.6244,0.6244
5,last15_press_events,last15_press_uplift,0.5557,0.5557
6,cumul_press_turnovers,cumul_pressing_others,0.5405,0.5405
7,cumul_pressing_others,pressing_minus_pressed,0.5327,0.5327
8,cumul_press_events,cumul_pressing_others,0.5317,0.5317
9,cumul_press_events,pressing_minus_pressed,-0.4336,0.4336


In [40]:
!pip install statsmodels

  Using cached statsmodels-0.14.6-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp312-cp312-win_amd64.whl (9.5 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
"""Cluster-robust GLM + BH-FDR."""
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


def cluster_robust_p(df: pd.DataFrame, feature: str) -> tuple[float, float, int]:
    sub = df[[feature, "scored_after", "fixture_id"]].dropna()
    if sub[feature].std() == 0 or len(sub) < 30:
        return float("nan"), float("nan"), len(sub)
    try:
        m_ = smf.logit(f"scored_after ~ {feature}", data=sub).fit(
            disp=False, maxiter=200,
            cov_type="cluster", cov_kwds={"groups": sub["fixture_id"]},
        )
        return float(m_.params[feature]), float(m_.pvalues[feature]), len(sub)
    except Exception:
        return float("nan"), float("nan"), len(sub)


rows = []
for f in NUMERIC:
    coef, p, n = cluster_robust_p(m, f)
    r = m[[f, "scored_after"]].corr().iloc[0, 1]
    rows.append({"feature": f, "n": n, "pearson_r": r, "coef": coef, "p_raw": p})
sig = pd.DataFrame(rows)

mask = sig["p_raw"].notna()
adj = np.full(len(sig), np.nan)
if mask.sum():
    _, adj_p, _, _ = multipletests(sig.loc[mask, "p_raw"].values, method="fdr_bh")
    adj[mask.values] = adj_p
sig["p_bh"] = adj
sig["bh_q05"] = sig["p_bh"] < 0.05
sig.sort_values("p_bh").reset_index(drop=True)


,feature,n,pearson_r,coef,p_raw,p_bh,bh_q05
0,top_third_press_share,3238,0.1075,1.5120,0.0007,0.0082,True
1,press_accuracy,3225,-0.0802,-1.2603,0.0032,0.0192,True
2,press_turnover_rate,3238,0.0694,1.1154,0.0053,0.0212,True
3,last15_press_events,3324,0.0518,0.0752,0.0085,0.0254,True
4,cumul_press_top_third,3324,0.0499,0.0498,0.0925,0.1586,False
5,mean_abs_pass_angle,2992,-0.0421,-0.0120,0.0845,0.1586,False
6,last15_press_uplift,3324,0.0268,0.1920,0.0877,0.1586,False
7,cumul_press_turnovers,3324,0.0352,0.0428,0.2548,0.3717,False
8,forward_pass_share,2992,-0.0376,-0.5693,0.3098,0.3717,False
9,pressing_minus_pressed,3324,0.0287,0.0165,0.2986,0.3717,False


### D - findings

Decisions:

* Pairs with `|pearson| >= 0.95` -> drop the dependent partner.
* BH q=0.05 survivors go to the manifest unconditionally.
* Features with negative coefficients but plausible mechanism (e.g.
  `press_turnover_rate`) are kept - direction matters and is informative.


## Section E - Final manifest

In [ ]:
"""Programmatic manifest."""

def best_non_baseline_bin(df: pd.DataFrame, feature: str) -> tuple[float, float, float, int]:
    sub = df[df["feature"] == feature]
    if sub.empty:
        return (float("nan"),) * 3 + (0,)
    sub = sub.sort_values("rate", ascending=False).iloc[0]
    return float(sub["rate"]), float(sub["ci_lo"]), float(sub["ci_hi"]), int(sub["n"])


rows = []
for feat_name in NUMERIC:
    rate, lo, hi, n = best_non_baseline_bin(binned_df, feat_name)
    rows.append({"feature": feat_name, "best_rate": rate,
                 "ci_lo": lo, "ci_hi": hi, "n": n})

manifest = pd.DataFrame(rows).merge(
    sig[["feature", "pearson_r", "p_bh", "bh_q05"]], on="feature", how="left"
)


def verdict(row) -> str:
    """Robust verdict that ignores spurious BH p_bh = 0 from perfect separation."""
    if pd.isna(row["best_rate"]):
        return "drop (no data)"
    rate_above = row["best_rate"] > BASELINE * 1.2
    ci_above = (not pd.isna(row["ci_lo"]) and row["ci_lo"] > BASELINE)
    if row["bh_q05"] is True and (rate_above or ci_above):
        return "KEEP"
    if ci_above:
        return "keep (cautious)"
    if row["best_rate"] > BASELINE * 1.4 and row["n"] >= 30:
        return "keep (cautious)"
    return "drop (flat)"


manifest["verdict"] = manifest.apply(verdict, axis=1)
manifest.sort_values(["verdict", "p_bh"]).reset_index(drop=True)


,feature,best_rate,ci_lo,ci_hi,n,pearson_r,p_bh,bh_q05,verdict
0,top_third_press_share,0.1067,0.0836,0.1352,553,0.1075,0.0082,True,KEEP
1,press_accuracy,0.1075,0.0707,0.1602,186,-0.0802,0.0192,True,KEEP
2,press_turnover_rate,0.0981,0.0793,0.1208,795,0.0694,0.0212,True,KEEP
3,last15_press_events,0.0766,0.0566,0.1030,509,0.0518,0.0254,True,KEEP
4,cumul_press_top_third,0.0713,0.0528,0.0956,561,0.0499,0.1586,False,drop (flat)
5,last15_press_uplift,0.0680,0.0580,0.0795,2118,0.0268,0.1586,False,drop (flat)
6,cumul_press_turnovers,0.0756,0.0546,0.1037,450,0.0352,0.3717,False,drop (flat)
7,pressing_minus_pressed,0.0738,0.0552,0.0979,583,0.0287,0.3717,False,drop (flat)
8,cumul_pressing_others,0.0654,0.0538,0.0793,1453,0.0024,0.9068,False,drop (flat)
9,mean_abs_pass_angle,0.0859,0.0652,0.1124,547,-0.0421,0.1586,False,keep (cautious)


: 

### Closing notes

* The manifest above is the deliverable. Once finalised, add a
  `features/pressure.py` module mirroring `features/shots.py` with a
  single `build_pressure_features(press_df, main_df) -> DataFrame` entry.
* The pressing-side variables (`cumul_pressing_others`,
  `pressing_minus_pressed`) leverage `pressing_player_appearance_id` -
  this is the only place in the dataset that lets us see the *opposite*
  side of a press event. If they survive the manifest they directly
  answer RQ4 "does pressure data help" with affirmative defensive
  signal that no other table carries.
* Any feature that wins here (especially `top_third_press_share` /
  `forward_pass_share`) becomes a strong candidate for cross-features
  with shots / runs (e.g. `top_third_press_share x set_piece_share`,
  analogous to the cross-table candidates we built for runs).
